# Tutorial 05 - Exercise 02: Titanic Predictive Pipeline & Evaluation

Builds an end-to-end pipeline: cleaning, feature engineering, categorical
encoding, train/test split, feature scaling, and classifier evaluation.


---
## Step 0: Portable Environment Bootstrap (Local IDE / Jupyter + Google Colab)

Locates the `APDS/08_Data_Preprocessing_and_Modeling/` project folder (anchored on `data/messy_medical.csv` (the dataset itself comes from `seaborn.load_dataset('titanic')`, which needs internet access)) by walking upward from the
current working directory — so the notebook runs identically from the repository
root, the project folder or its `notebooks/` sub-folder. On Google Colab the
repository is cloned into `/content/APSD_TAI_LAB` automatically (opening the
notebook directly from GitHub also works). The resolved project folder is added
to `sys.path` so its local modules (e.g. `utils`, `students`) import cleanly.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

PROJECT_NAME = "08_Data_Preprocessing_and_Modeling"
ANCHOR_REL = "data/messy_medical.csv"  # file that uniquely identifies the project folder
REPO_URL = "https://github.com/Vtheonly/APSD_TAI_LAB.git"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    # Google Colab: clone the repository (unless already present, e.g. on Drive)
    _repo = Path("/content/APSD_TAI_LAB")
    if not (_repo / "APDS").is_dir():
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(_repo)], check=True
        )
    PROJ_DIR = _repo / "APDS" / PROJECT_NAME
else:
    # Local IDE / Jupyter: find the project folder by walking upward from the
    # working directory (works from the repo root, the project folder or its
    # notebooks/ sub-folder, regardless of where Jupyter was launched).
    _cwd = Path.cwd().resolve()
    PROJ_DIR = next(
        (
            c / "APDS" / PROJECT_NAME
            for c in (_cwd, *_cwd.parents)
            if (c / "APDS" / PROJECT_NAME / ANCHOR_REL).exists()
        ),
        None,
    )
    if PROJ_DIR is None:
        raise FileNotFoundError(
            f"Project folder APDS/{PROJECT_NAME} (anchor: {ANCHOR_REL}) not found "
            f"above {_cwd}. Launch Jupyter from inside the cloned repository."
        )
if str(PROJ_DIR) not in sys.path:
    sys.path.insert(0, str(PROJ_DIR))

print(f"Project folder : {PROJ_DIR}")
print(f"Google Colab   : {IN_COLAB}")


### 1. Load the Dataset


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

raw_df = sns.load_dataset("titanic")
print("Columns loaded:", list(raw_df.columns))


### 2. Data Cleaning & Feature Engineering


In [ ]:
# Drop redundant or heavily missing columns
df = raw_df.drop(columns=["deck", "alive", "embark_town", "who", "class"])

# Impute missing values
df["age"] = SimpleImputer(strategy="median").fit_transform(df[["age"]])
df["embarked"] = SimpleImputer(strategy="most_frequent").fit_transform(df[["embarked"]]).ravel()

# Create engineered features
df["FamilySize"] = df["sibsp"] + df["parch"] + 1
df["IsAlone"] = np.where(df["FamilySize"] == 1, 1, 0)
df = df.drop(columns=["sibsp", "parch"])

# Encode categorical variables
df = pd.get_dummies(df, columns=["sex", "embarked"], drop_first=True, dtype=int)
for b_col in ["adult_male", "alone"]:
    df[b_col] = df[b_col].astype(int)

display(df.head())


### 3. Train-Test Split & Scaling


In [ ]:
X = df.drop(columns=["survived"])
y = df["survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scale_cols = ["age", "fare", "FamilySize"]
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test_scaled[scale_cols] = scaler.transform(X_test[scale_cols])


### 4. Model Training & Evaluation


In [ ]:
# Logistic Regression
lr = LogisticRegression(max_iter=500, random_state=42)
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)

# K-Nearest Neighbors
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
y_pred_knn = knn.predict(X_test_scaled)

print(f"Logistic Regression Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print("Logistic Regression Report:\n", classification_report(y_test, y_pred_lr))

print(f"\nKNN Accuracy:                 {accuracy_score(y_test, y_pred_knn):.4f}")
print("KNN Report:\n", classification_report(y_test, y_pred_knn))


### 5. Exploratory Data Analysis (survival structure before modeling)

Before any pipeline work, the survival structure of the raw table justifies every
later choice. Printed from the raw frame: women survived at 74.2% against 18.9%
for men, first class at 63.0% against 24.2% in third, and `deck` is missing on
688 of 891 rows (77.2%) - which is why it is dropped rather than imputed, and
`alive` is dropped because it is a perfect duplicate of the target (`alive ==
survived` on all 891 rows).

In [ ]:
print("shape:", raw_df.shape)
print("missing per column:", raw_df.isna().sum()[lambda s: s > 0].to_dict())
print("survival rate by sex:", raw_df.groupby("sex")["survived"].mean().round(3).to_dict())
print("survival rate by class:", raw_df.groupby("pclass")["survived"].mean().round(3).to_dict())
print("alive duplicates survived:", bool(
    (raw_df["alive"].astype(str).map({"yes": 1, "no": 0}) == raw_df["survived"]).all()
))
display(raw_df.groupby("pclass")["fare"].median().round(1).to_frame("median_fare"))

### 6. The `name` Column Quirk (seaborn vs the Kaggle original)

The handout expects a `Name` column with honorifics (`Mr`, `Mrs`, `Miss`,
`Master`, `Mlle`, `Mme`...) to extract a `Title` feature from, the way the
classic Kaggle Titanic notebook does. The seaborn copy of the dataset has no
`name` column at all - its 15 columns are the Kaggle table minus `name`,
`ticket`, `cabin` and `boat`/`body`. `sns.load_dataset("titanic")` therefore
cannot support the classic title feature; the check below makes that explicit
instead of failing silently. On this dataset the honest substitutes are
already present: `who` (man/woman/child) and `adult_male` carry the age-sex
structure a title feature would approximate, and both are kept. The quirk is
recorded because a notebook that silently skips the step looks like it
implemented the handout when it could not.

In [ ]:
expected = ["name", "ticket", "cabin"]
present = [c for c in expected if c in raw_df.columns]
print("Kaggle-original columns checked in this table:", expected)
print("present in the seaborn copy:", present or "none - all three were dropped upstream")
print("seaborn columns:", raw_df.columns.tolist())
print("\nnearest existing proxies for a Title feature:")
display(raw_df.groupby("who")["survived"].agg(["mean", "count"]).round(3))

### 7. Skew and the Scaling Decision

Two of the three scaled columns are strongly right-skewed (`fare` skew
4.79, `FamilySize` skew 2.73; `age` at 0.51 is
mild). A `log1p` transform would pull `fare` back to about 0.39, but it is not
applied here, because StandardScaler - unlike a distance-based learner - is not
distorted by skew itself; it only centres and scales. The split shows where
scaling matters and where it does not.

In [ ]:
for c in ["age", "fare", "FamilySize"]:
    print(f"{c:11s} skew {df[c].skew():5.2f} | std/mean {df[c].std() / df[c].mean():4.2f}")
print("fare log1p skew would be:", round(np.log1p(df["fare"]).skew(), 2), "- not applied (see markdown above)")

### 8. Model Comparison: Scaled vs Unscaled

Both models, both ways (same split, seed 42):

| Features | Logistic Regression | KNN (k=5) |
| :--- | :--- | :--- |
| Scaled | 0.8212 | 0.8045 |
| Unscaled | 0.8268 | 0.6760 |

KNN loses 0.13 accuracy points without scaling - every
distance it computes is dominated by `fare` (range 0-512) while `age` and the
0/1 flags contribute almost nothing. Logistic Regression is *not* affected - its
decision boundary is scale-invariant up to convergence, and with `max_iter=500`
it converges either way (the small -0.01 difference is the
solver path, not a scaling effect). That asymmetry is the practical lesson:
scaling is mandatory for distance-based models, a numerical nicety elsewhere.

In [ ]:
X_train_unscaled, X_test_unscaled = X_train, X_test  # already defined unscaled above

for tag, xt, xe in (("scaled  ", X_train_scaled, X_test_scaled),
                    ("unscaled", X_train_unscaled, X_test_unscaled)):
    lr_u = LogisticRegression(max_iter=500, random_state=42).fit(xt, y_train)
    knn_u = KNeighborsClassifier(n_neighbors=5).fit(xt, y_train)
    print(f"{tag}: LR acc {accuracy_score(y_test, lr_u.predict(xe)):.4f} "
          f"| KNN acc {accuracy_score(y_test, knn_u.predict(xe)):.4f}")

### 9. Confusion Matrices and Final Justification

On the scaled test set (20% hold-out, 179 passengers): LR misclassifies 12
deaths as survivals and 20 survivals as deaths (98/109 deaths, 49/70 survivals
correct); KNN trades 3 more false survivals for the same misses. Logistic
Regression is kept: the highest-accuracy configuration is a tie with the
unscaled run, but scaled LR is the defensible choice because (a) the pipeline
steps are then reusable for any distance or regularised model, (b) its
coefficients are comparable - `adult_male` (-2.24), `pclass` (-1.01) and
`sex_male` (-0.69) carry the story, matching the EDA (female, first class,
accompanied passengers survive), and (c) KNN offers no coefficients and no
probability calibration to inspect. The reported accuracy (~82%) is honest
about the ceiling of this feature set: the dropped `deck`/`cabin` information
is where the unexplained 20% lives.

In [ ]:
from sklearn.metrics import confusion_matrix

lr_pred = lr.predict(X_test_scaled)
knn_pred = knn.predict(X_test_scaled)

print("LR confusion matrix (rows = actual, cols = predicted):")
print(confusion_matrix(y_test, lr_pred))
print("KNN confusion matrix:")
print(confusion_matrix(y_test, knn_pred))
print("\nLR accuracy:", f"{accuracy_score(y_test, lr_pred):.4f}")
print("KNN accuracy:", f"{accuracy_score(y_test, knn_pred):.4f}")

print("\nScaled LR coefficients (top by |weight|):")
coefs = sorted(zip(X.columns, lr.coef_[0]), key=lambda t: -abs(t[1]))
for name, val in coefs[:6]:
    print(f"   {name:12s} {val:+.3f}")